## 6.1 k-Means with scikit-learn

[Lecture 5](https://colab.research.google.com/drive/1zSmntNtn9EnFqObH8-5Im_z361yS1Xev?usp=sharing) and [Homework 5](https://colab.research.google.com/drive/1nm-lpzl7B3cKvyuu2160WXcs7d7L0iJJ?usp=sharing) focused on learning about the k-Means Clustering Algorithm by running the steps "from scratch." In this lecture we'll focus on using the [scikit-learn library](https://scikit-learn.org/stable/) to do the work for us.

```sklearn``` is a machine learning library for the Python programming language. It's built "on-top" of more basic libraries like ```numpy``` and it allows us to easily implement a large number of popular machine learning models. We have used it before in this class: where and why?


**Example**: Use the ```cluster``` submodule from the ```sklearn``` library to group the vectors from the example in [Lecture 5](https://colab.research.google.com/drive/1zSmntNtn9EnFqObH8-5Im_z361yS1Xev?usp=sharing).

In [ ]:
import numpy as np

X = np.array([[1, 1], [1,0], [0, 2], [2, 4], [3, 5]])
# Note that we don't need to manually break this up into x_i row vectors like we did from scratch; scikit-learn will do it for us.

In [ ]:
from sklearn import cluster

kmeans_ex = cluster.KMeans(n_clusters = 2, n_init = 10, random_state = 1)
# This creates the cluster object and initializes it for the k-Means model we'll build in the next step.

What does ```n_init``` do? Since we are randomly choosing the group vector representatives it is possible that the final groupings will be dependent on these random initial choices. This tends to be an issue when the data vectors are close together (groups aren't well-seperated) and/or with high dimensional datasets.

The following definition introduces a commonly used technique for handling this potential problem.

**Definition**. An <u>ensemble model</u> is a machine learning system that is constructed with a set of individual models working in parallel, whose outputs are combined in a predetermined way to produce a single answer for a given problem.

**Questions**. How many different k-Means models are being constructed in the code cell above? Suppose the final groupings of these models don't agree? What is a reasonable way to make a final group assignment?

**Note**. In 2003 Charles Elkan published a [paper](https://cdn.aaai.org/ICML/2003/ICML03-022.pdf) called *Using the Triangle Inequality to Accelerate k-Means*. This is version of the k-Means algorithm is harder to understand (less intuitive to the new practitioner) but less computationially expensive than Lloyd's. As is common with highly optimized code libraries, scikit-learn prioritizes computational efficiency over interpretability.

**Question**. Other than the fact this is scientifically interesting, why else might we care about this?

In [ ]:
# Run the algorithm- this is where we feed it the data and it:
# a) learns the optimal group representative vectors (fit) and
# b) "predicts" which group the data vectors belong to.
# Recall groups from last time: {x_0, x_1, x_2}, {x_3, x_4}

labels = kmeans_ex.fit_predict(X)
print(labels)
type(labels)

**Questions**. Does this result agree with what we got from scratch? What kind of Python object is ```sklearn``` using to represent the final groupings?

In [ ]:
import matplotlib.pyplot as plt

# Get unique labels for plotting purposes
u_labels = np.unique(labels)

#plot
centers = np.array(kmeans_ex.cluster_centers_)
for i in u_labels:
    plt.scatter(X[labels == i , 0] , X[labels == i , 1] , label = i)
plt.scatter(centers[:,0], centers[:,1], marker="x", color='k')
plt.legend()
plt.show()

In [ ]:
# Use results to make predictions about observations
new_data = np.array([[0, 0], [12, 3]]) # This array creates two "new" observations that our trained model can now assign to group(s).
kmeans_ex.predict(new_data) # We now make the prediction by applying the kmeans_ex model's predict function to the new data.

### Summary: Why Model?

1. We want to group the observations we already have: learn something about the relationships between them. Think ```fit_predict``` method.

2. We want to know which group a new observation (that the model didn't train on) would be in. Think ```predict``` method.

## 6.2 k-Means and the Iris Dataset

**Example**: Use ```sklearn``` to group the data in the ```iris``` dataset.

In [ ]:
# Upload file.
from google.colab import files

uploaded = files.upload()

In [ ]:
# View with Pandas.
import pandas as pd

iris_df = pd.read_csv("iris.csv")
iris_df.head()

In [ ]:
# Convert to a numpy ndarray.
X = iris_df.to_numpy()

In [ ]:
# Standardize the data: Why do we need to do this for iris when we didn't do it for the previous example?
from sklearn import preprocessing

scaler = preprocessing.StandardScaler()
Z = scaler.fit_transform(X[:,0:4])

In [ ]:
# Run k-Means. How many groups should we choose?
from sklearn import cluster

kmeans_iris = cluster.KMeans(n_clusters=3, n_init = 10, random_state=0)
labels = kmeans_iris.fit_predict(Z)
print(labels)

In [ ]:
# Plot the groups. What are we looking at, specifically, with this particular plot?
import matplotlib.pyplot as plt

u_labels = np.unique(labels)
centers = np.array(kmeans_iris.cluster_centers_)
for i in u_labels:
    plt.scatter(Z[labels == i , 0] , Z[labels == i , 1], label = i)
plt.scatter(centers[:,0], centers[:,1], marker="x", color='k')
plt.legend()
plt.show()


**Question**. Why do the groups appear to overlap? What does this indicate about the way the K-Means algorithm grouped the irises? Explore this from several different perspectives by changing which vectors we're plotting. What if we did $k=2$? How could we interpret this?

## 6.3 Principle Component Analysis

Principle Component Analysis (PCA) is a *dimension reduction* techniqe. The mathematics (described in Chapter 15 of *Practical Linear Algebra for Data Science*) involve some advanced matrix algebra, but the idea is straightforward: project a vector (or data point) from a higher dimensional space to a lower dimensional space. More specifically, the direction of largest variation (equivalently, largest standard deviation) becomes the "first principal component" of the dataset. The second principal component is found by considering all possible directions that are orthogonal to the first principal component and then choosing the direction that has the highest standard deviation. And so on. For visualization purposes it is common to just use the first two principle components (why?).

Let's use the ```PCA``` module from the ```sklearn.decomposition``` library to reduce the ```iris``` dataset to 2 dimensions. Then we'll run the K-Means algorithm on these two dimensions.

In [ ]:
from sklearn import decomposition

pca = decomposition.PCA(2) # Initialize the PCA for reducing to 2 dimensions.
Y = pca.fit_transform(Z) # Run the PCA dimension reduction algorithm on the standardized iris dataset and name the resulting transformed dataset Y.
print(Y[0:10,:]) # How many columns do you expect to see?

Now run k-Means again on the transformed dataset ```Y```. The plotted results are what is most interesting here.

In [ ]:
from sklearn import cluster

kmeans_iris_pca = cluster.KMeans(n_clusters=3, n_init = 10, random_state=0)
labels_pca = kmeans_iris_pca.fit_predict(Y)

In [ ]:
import matplotlib.pyplot as plt

u_labels = np.unique(labels_pca)

for i in u_labels:
    plt.scatter(Y[labels_pca == i , 0] , Y[labels_pca == i , 1], label = i)
plt.legend()
plt.show()

## 6.4 Real-World Use-Cases

Discuss the following scenarios: how could k-Means be used to an organization's advantages?

* An online bookseller wants to know which books to "recommend" to a customer, given the book they just purchased.

* A credit card company employs a "rapid response team" to identify and track fraud in real-time. They could do things like randomly contact card-holders and ask about payments. What would be an improvement over this?

* A small company has a customer service center that employs two full-time agents to answer questions about the company's products and troubleshoot problems. To get an agent on the phone a customer fills out an online questionaire that collects data about the problem(s) they are having with the product.